# Lab 2: Compressing a jet tagger
**AIMS Edge ML for Science · Day 4 · two sessions of ~2 hours**

On Day 3 you saw *how* quantization, pruning and distillation work. Today you **use** them, with a real tool (Brevitas), on the jet tagger from Lab 1.
You never write a quantizer: you choose the bits, the tool does the rest. The questions are about physics and hardware:
when does the tagger stop being good enough, which part of the network is fragile, which jets suffer first, and what is the cheapest tagger that still does the job?

| | Section | Time |
|---|---|---|
| **Part A (morning)** | 0. Setup and the fp32 tagger | 15 min |
| *post-training quantization* | A1. Quantize in one line, sweep the bits | 20 min |
| | A2. Which layer is fragile? · ✋ Pause | 30 min |
| | A3. Which jets suffer? | 20 min |
| | A4. Your turn: the best tagger under a BOPs budget · ✋ Pause | 25 min |
| ☕ **Break** | | |
| **Part B (afternoon)** | B1. Training with quantization (QAT) | 25 min |
| *training for the hardware* | B2. Protect the fragile layer | 10 min |
| | B3. Pruning: how many weights can go? | 20 min |
| | B4. Distillation: a big teacher for a small student | 30 min |
| | B5. Everything on one plot: the performance–BOPs front · ✋ Pause | 20 min |
| | B6. Save a model for Lab 3 | 5 min |

| Mark | Meaning |
|---|---|
| ✏️ **Yours to change** | A cell with values you can edit. Edit, then rerun that cell (and the plot cell after it, if there is one). |
| 💭 **Think about** | A question to discuss with your neighbour. Nothing to type. |
| ✋ **Pause** | We stop and discuss as a class. |
| 📌 **What this shows** | The idea behind what you just ran. |
| 🟩 **AI-on** · 🟥 **AI-off** | The whole lab is 🟩 AI-on, except the one 🟥 "guess first" in A2: guess on your own, then run. |

> **How to use this lab.** Run the cells in order (Shift + Enter). Cells marked ✏️ are yours to change: edit the value, then rerun that cell and the plot cell below it.
> Each rerun **overwrites** the previous output, but every model you score is also added to a running table, `RESULTS`, so nothing you try is lost:
> all of it ends up on the plot in B5.

> **Before you start:** **File → Save a copy in Drive** (work in the copy), then run the first code cell. Any runtime is fine; this lab doesn't need the GPU.

## 0. Setup and the fp32 tagger

In [ ]:
# --- Colab setup: run once per session (~1-2 min: it also installs Brevitas) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the dataset on your Google Drive between sessions
LAB = "lab02_quantization"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "brevitas"], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
        os.environ["SCIKIT_LEARN_DATA"] = "/content/drive/MyDrive/fastml_data/sklearn"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
sys.path.append("..")
import numpy as np, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from common.jets import load_jets, CLASSES
from common.compress import (SIZES, mlp, qat_mlp, train, accuracy, bkg_rejection, logits, ptq, bops, per_class_accuracy, confusion,
                             prune_global, sparsity_per_layer, snapshot)

torch.manual_seed(0); np.random.seed(0)
X_train, X_test, y_train, y_test, FEATURES = load_jets()
idx = np.random.default_rng(0).permutation(len(X_train))[:200_000]          # a subsample, so every training run takes about a minute
Xtr, ytr = torch.from_numpy(X_train[idx]), torch.from_numpy(y_train[idx])
Xte, yte = torch.from_numpy(X_test[:50_000]), torch.from_numpy(y_test[:50_000])
CAL = Xtr[:20_000]                                                          # calibration data for post-training quantization

def score(m):
    """The two numbers we care about: accuracy over all 5 classes, and the top tagger's background rejection at 50% efficiency."""
    return accuracy(m, Xte, yte), bkg_rejection(m, Xte, yte, sig=CLASSES.index("t"))

RESULTS = {}            # every model you score: name -> (accuracy, rejection, BOPs, kind). Plotted in B5.
def record(name, model, cfg, sizes=SIZES, sparsity=None, kind="other"):
    """Score a model, add it to RESULTS (same name = overwritten), print one line."""
    acc, rej = score(model)
    cost = bops(cfg, sizes=sizes, sparsity=sparsity)
    RESULTS[name] = (acc, rej, cost, kind)
    print(f"{name:<34} accuracy {acc:.4f} | rejection {rej:6.0f} | BOPs {cost/1e3:9.1f}k")
    return acc, rej

def fmt(cfg):
    """[(6, 6), (3, 4), ...] -> '6/6 3/4 ...' (weight bits / input bits, per layer)"""
    return " ".join(f"{w}/{a}" for w, a in cfg)

t0 = time.perf_counter()
base = train(mlp(), Xtr, ytr, epochs=4, seed=0)                            # the fp32 jet tagger, 16 -> 64 -> 32 -> 32 -> 5
ACC32, REJ32 = record("fp32", base, [(None, None)] * 4, kind="fp32")
print(f"({time.perf_counter()-t0:.0f} s)")

**The physics metric.** In a trigger you'd use this tagger to *keep top-quark jets* and *throw away everything else*. What matters is how much background you let through
at a fixed signal efficiency: the **background rejection at 50% top efficiency** = 1 / (fraction of non-top jets that pass when half the top jets pass).
A rejection of 400 means 1 background jet in 400 sneaks through. We track it alongside plain accuracy.

**The hardware metric.** **BOPs** (bit operations) = for every multiplication in the network, (bits of one operand) × (bits of the other), summed. For a dense layer that is
MACs × weight bits × input bits. It's a proxy for how much chip area the arithmetic takes (Day 3).

In [ ]:
def top_roc(m):
    p = torch.softmax(logits(m, Xte), 1)[:, CLASSES.index("t")].numpy()
    fpr, tpr, _ = roc_curve(yte.numpy() == CLASSES.index("t"), p)
    return tpr, 1 / np.maximum(fpr, 1e-6)

plt.semilogy(*top_roc(base), label="fp32"); plt.axvline(0.5, c="gray", ls=":")
plt.xlabel("top-jet efficiency"); plt.ylabel("background rejection"); plt.title("Top tagger, fp32"); plt.legend(); plt.grid(alpha=.3); plt.show()

---
# Part A (morning): post-training quantization
## A1. Quantize in one line, sweep the bits (20 min) 🟩
`ptq(model, cfg, CAL)` takes the trained fp32 model and returns a quantized copy. `cfg` lists, for each of the 4 layers, **(weight bits, input bits)**.
Brevitas chooses the scales and calibrates the activation ranges on `CAL`. It uses power-of-two steps, i.e. exactly the `ap_fixed` numbers an FPGA uses (Lab 3 converts these very models).

The cell below quantizes every layer to the same number of bits, for each value in `BITS`.

In [ ]:
# ✏️ BITS: the bit widths to try (same for weights and inputs, in every layer). Add or remove values and rerun this cell and the next.
BITS = [8, 6, 5, 4, 3, 2]

for b in BITS:
    record(f"PTQ {b} bits", ptq(base, [(b, b)] * 4, CAL), [(b, b)] * 4, kind="PTQ")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(BITS, [RESULTS[f"PTQ {b} bits"][0] / ACC32 for b in BITS], "o-", label="accuracy")
ax[0].plot(BITS, [RESULTS[f"PTQ {b} bits"][1] / REJ32 for b in BITS], "s-", label="top-tag rejection")
ax[0].set(xlabel="bits (weights and inputs, every layer)", ylabel="fraction of fp32 value", title="What survives quantization"); ax[0].legend(); ax[0].grid(alpha=.3)
for b in BITS[:3]:
    ax[1].semilogy(*top_roc(ptq(base, [(b, b)] * 4, CAL)), label=f"{b} bits")
ax[1].semilogy(*top_roc(base), "k--", label="fp32"); ax[1].axvline(0.5, c="gray", ls=":")
ax[1].set(xlabel="top-jet efficiency", ylabel="background rejection", title="Top tagger ROC"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

💭 **Think about:** At how many bits does the tagger stop being good enough? Does the accuracy or the rejection fall first?
If you were building a trigger and someone showed you only the accuracy curve, would you have chosen the same number of bits?

📌 **What this shows.** Accuracy counts every jet equally. The rejection lives in the *tail* of the top-score distribution: the few background jets that score almost as high as
real tops. Small rounding errors move exactly those jets across the cut, so the two metrics need not fall together. For a trigger, the rejection is the one that matters.

## A2. Which layer is fragile? (30 min) ✋ Pause
Now squeeze **one layer at a time** to 3 bits while the others stay at 8. Do it three ways: weights and inputs together, only the weights, and only the inputs.

🟥 **Guess first (1 min, no tools).** Which of the four layers will hurt the rejection most when squeezed: 16→64, 64→32, 32→32 or 32→5? Weights or inputs?
Hint: think about what each layer *receives*. The first layer gets the raw physics features. Tell your neighbour, then run.

In [ ]:
# ✏️ SQUEEZE: the bit width for the one squeezed layer (try 4 or 2 as well).
SQUEEZE = 3

LAYERS = ["16→64", "64→32", "32→32", "32→5"]
rows = []
for i in range(4):
    out = {}
    for label, pair in [("weights + inputs", (SQUEEZE, SQUEEZE)), ("weights only", (SQUEEZE, 8)), ("inputs only", (8, SQUEEZE))]:
        cfg = [(8, 8)] * 4; cfg[i] = pair
        out[label] = score(ptq(base, cfg, CAL))[1] / REJ32
    rows.append(out)
    print(f"layer {i} ({LAYERS[i]:>5}) at {SQUEEZE} bits: " + " | ".join(f"{k}: {v:5.1%}" for k, v in out.items()) + "   (rejection, fraction of fp32)")

x = np.arange(4); w = 0.27
for j, label in enumerate(["weights + inputs", "weights only", "inputs only"]):
    plt.bar(x + (j - 1) * w, [r[label] for r in rows], w, label=label)
plt.xticks(x, [f"layer {i}\n{LAYERS[i]}" for i in range(4)]); plt.ylabel("top-tag rejection, fraction of fp32")
plt.title(f"One layer at {SQUEEZE} bits, the rest at 8"); plt.legend(); plt.grid(alpha=.3, axis="y"); plt.show()

💭 **Think about:** Was your guess right? Which layer is the fragile one, and is it its weights or its inputs? Now look at what that layer receives:

In [ ]:
show = list(range(min(6, len(FEATURES))))
fig, ax = plt.subplots(1, len(show), figsize=(3 * len(show), 2.8))
for a, k in zip(ax, show):
    a.hist(X_train[:200_000, k], bins=100, log=True); a.set_title(str(FEATURES[k]), fontsize=9)
plt.suptitle("Input features (standardized), log scale"); plt.tight_layout(); plt.show()
print("largest |value| per feature, in standard deviations:", np.round(np.abs(X_train[:200_000]).max(0), 1))

📌 **What this shows.** Many jet-substructure features have **long tails**: a few jets sit many standard deviations out (large masses, many constituents).
A quantizer has to cover the largest value it sees, so with few bits most of the levels are spent on the rare tail and the bulk of the jets gets only one or two.
Hidden layers see ReLU outputs, which are better behaved. If your bar chart singles out one layer, ask whether this is the reason.

💭 **Think about:** You have a fixed number of bits to hand out. Would you spend them where most of the arithmetic is (the 64→32 layer has the most multiplications), or where the information enters?

## A3. Which jets suffer? (20 min) 🟩
Compare per-class accuracy of the fp32 tagger with a uniformly quantized one, and see where the extra mistakes go.

In [ ]:
# ✏️ COMPARE_BITS: the bit width of the quantized tagger to compare with fp32 (try 5, 4, 3).
COMPARE_BITS = 4

qc = ptq(base, [(COMPARE_BITS, COMPARE_BITS)] * 4, CAL)
pc32, pcq = per_class_accuracy(base, Xte, yte), per_class_accuracy(qc, Xte, yte)
x = np.arange(5)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(x - 0.2, pc32, 0.4, label="fp32"); ax[0].bar(x + 0.2, pcq, 0.4, label=f"{COMPARE_BITS} bits")
ax[0].set_xticks(x, ["gluon", "light quark", "W", "Z", "top"]); ax[0].set_ylabel("accuracy for that class"); ax[0].legend(); ax[0].grid(alpha=.3, axis="y")
D = confusion(qc, Xte, yte) - confusion(base, Xte, yte)          # where the quantized model makes MORE mistakes than fp32
im = ax[1].imshow(D, cmap="RdBu_r", vmin=-abs(D).max(), vmax=abs(D).max()); plt.colorbar(im, ax=ax[1], label=f"change in fraction ({COMPARE_BITS} bits − fp32)")
ax[1].set_xticks(x, ["g", "q", "W", "Z", "t"]); ax[1].set_yticks(x, ["g", "q", "W", "Z", "t"]); ax[1].set(xlabel="predicted", ylabel="true", title="Extra confusion")
plt.tight_layout(); plt.show()

💭 **Think about:** Which class loses the most, and what is it mistaken for? Does that pair make physical sense?
If your analysis needed W bosons rather than tops, would you quantize the tagger the same way?

📌 **What this shows.** W and Z bosons both decay to two quarks and give two-prong jets; their masses differ by only ~11 GeV (80 vs. 91 GeV). Telling them apart relies on
small differences in the features, which is exactly what coarse rounding blurs. Quantization need not hurt all physics equally.

## A4. Your turn: the best tagger under a BOPs budget (25 min) ✋ Pause 🟩
An FPGA has room for **80k BOPs**. Uniform 4 bits costs 68k; uniform 6 bits (153k) doesn't fit.
Choose your own bits per layer to get the **highest top-tag rejection** within the budget, using what you found in A2 and A3.

✏️ Edit `MY_CFG` and rerun the cell, as many times as you like. Every try that fits is added to `RESULTS` under its own name, and the cell after it lists all your tries so far.
At the ✋ we compare the best rejection in the room.

In [ ]:
BUDGET = 80_000
# ✏️ MY_CFG: (weight bits, input bits) for each of the 4 layers. Edit, rerun, repeat.
MY_CFG = [(4, 4), (4, 4), (4, 4), (4, 4)]

cost = bops(MY_CFG)
if cost > BUDGET:
    print(f"{fmt(MY_CFG)}: {cost:,.0f} BOPs is over the {BUDGET:,} budget. Take some bits away and rerun.")
else:
    record(f"PTQ {fmt(MY_CFG)}", ptq(base, MY_CFG, CAL), MY_CFG, kind="PTQ mixed")

In [ ]:
# Your tries so far, best rejection first
tries = sorted([(v[1], k, v[2]) for k, v in RESULTS.items() if v[3] == "PTQ mixed"], reverse=True)
for rej, name, cost in tries:
    print(f"{name:<34} rejection {rej:6.0f} ({rej/REJ32:4.0%} of fp32) | BOPs {cost/1e3:5.1f}k")

💭 **Think about:** Where did your best configuration spend its bits, and where did it save them? Look at which layers have the most multiplications (16·64, 64·32, 32·32, 32·5)
and which one was fragile in A2. That idea, different bits for different layers, is **mixed precision**. In Part B you'll train for it instead of just rounding.

---
# ☕ Break: Part B starts here

**If your Colab session was disconnected over the break** (you'll see "Reconnect" or errors like `name 'base' is not defined`): click on this cell, then
**Runtime → Run before** (Ctrl + F8). That reruns everything above (~3 min, including the fp32 training) and you can carry on. Otherwise, just continue.

# Part B (afternoon): training for the hardware
## B1. Training with quantization (25 min) 🟩
`qat_mlp(cfg)` builds the same network with Brevitas quantizers *inside* it. Training it as usual (**quantization-aware training**, QAT) lets the weights adapt to the rounding.
The cell trains one QAT model per bit width (~1 min each) and compares it with this morning's PTQ at the same bits.

In [ ]:
# ✏️ QAT_BITS: bit widths to train (each takes ~1 min). If the room is slow, use [6, 4].
QAT_BITS = [8, 6, 4, 3]

qat = {}
for b in QAT_BITS:
    t0 = time.perf_counter()
    qat[b] = train(qat_mlp([(b, b)] * 4), Xtr, ytr, epochs=4, seed=0)
    record(f"QAT {b} bits", qat[b], [(b, b)] * 4, kind="QAT")
    if f"PTQ {b} bits" not in RESULTS:
        record(f"PTQ {b} bits", ptq(base, [(b, b)] * 4, CAL), [(b, b)] * 4, kind="PTQ")
    print(f"   ({time.perf_counter()-t0:.0f} s)")

In [ ]:
plt.plot(QAT_BITS, [RESULTS[f"PTQ {b} bits"][1] / REJ32 for b in QAT_BITS], "o-", label="PTQ (this morning)")
plt.plot(QAT_BITS, [RESULTS[f"QAT {b} bits"][1] / REJ32 for b in QAT_BITS], "s-", label="QAT")
plt.axhline(1, c="gray", ls=":"); plt.xlabel("bits (every layer)"); plt.ylabel("top-tag rejection, fraction of fp32"); plt.legend(); plt.grid(alpha=.3); plt.show()

💭 **Think about:** At which bit widths is the extra training worth it, and at which does PTQ already do the job? If you only had five minutes before a deadline, which would you use?

📌 **What this shows.** PTQ rounds weights that were never meant to be rounded. QAT trains *through* the rounding (with the straight-through estimator from Day 3), so the network can
move its weights to places where rounding hurts less. It costs a training run; whether it pays off depends on how hard you squeeze.

## B2. Protect the fragile layer (10 min) 🟩
Train a mixed-precision model: generous bits on the fragile layer from A2, few bits elsewhere. Compare it with uniform QAT at similar BOPs.

In [ ]:
# ✏️ MIXED: (weight bits, input bits) per layer. Try your best A4 configuration too.
MIXED = [(8, 8), (3, 3), (3, 3), (3, 3)]

qat["mixed"] = train(qat_mlp(MIXED), Xtr, ytr, epochs=4, seed=0)
record(f"QAT {fmt(MIXED)}", qat["mixed"], MIXED, kind="QAT mixed")
for b in [3, 4]:
    if f"QAT {b} bits" in RESULTS:
        acc, rej, cost, _ = RESULTS[f"QAT {b} bits"]
        print(f"   compare: QAT {b} bits uniform             accuracy {acc:.4f} | rejection {rej:6.0f} | BOPs {cost/1e3:9.1f}k")

💭 **Think about:** For about the same BOPs, which would you deploy, the uniform model or the mixed one? What does that tell you about where the information enters the network?

## B3. Pruning: how many weights can go? (20 min) 🟩
Remove the smallest weights of a QAT model (magnitude pruning), then fine-tune briefly so the remaining weights can compensate. The cell scans the **sparsity**,
the fraction of weights removed. A removed weight is a multiplier that is never built, so BOPs scale with (1 − sparsity) layer by layer.

In [ ]:
# ✏️ PRUNE_BITS: which QAT model from B1 to prune.  SPARSITIES: the fractions of weights to remove.
PRUNE_BITS = 6
SPARSITIES = [0.5, 0.75, 0.9, 0.95]

cfg = [(PRUNE_BITS, PRUNE_BITS)] * 4
for s in SPARSITIES:
    qp = prune_global(snapshot(qat[PRUNE_BITS]), s)
    train(qp, Xtr, ytr, epochs=2, lr=1e-3, seed=0)                 # fine-tune; pruned weights stay at zero
    sp = sparsity_per_layer(qp)
    record(f"QAT {PRUNE_BITS} + prune {s:.0%}", qp, cfg, sparsity=sp, kind="pruned")
    print(f"   zeros per layer: {[f'{z:.0%}' for z in sp]}")

In [ ]:
names = [f"QAT {PRUNE_BITS} bits"] + [f"QAT {PRUNE_BITS} + prune {s:.0%}" for s in SPARSITIES]
plt.plot([0] + SPARSITIES, [RESULTS[n][1] / REJ32 for n in names], "o-")
plt.axhline(1, c="gray", ls=":"); plt.xlabel("sparsity (fraction of weights removed)"); plt.ylabel("top-tag rejection, fraction of fp32")
plt.title(f"Pruning the {PRUNE_BITS}-bit QAT model"); plt.grid(alpha=.3); plt.show()

💭 **Think about:** Up to what sparsity is pruning almost free? Look at the zeros per layer: which layers did global pruning thin the most, and which did it leave alone?
Is that consistent with what you found about the fragile layer in A2?

📌 **What this shows.** Global magnitude pruning removes the smallest weights *wherever they are*, so each layer ends up with a different sparsity. On a CPU or GPU the zeros
are still stored and multiplied (no speed-up); on an FPGA, hls4ml simply doesn't build those multipliers.

## B4. Distillation: a big teacher for a small student (30 min) 🟩
Distillation trains a small model (the **student**) to match the outputs of a big, accurate model (the **teacher**), not just the true labels.
The teacher's softmax says *how* a jet looks, e.g. "70% W, 25% Z": that is more information per jet than the single true label.

First, train the teacher: a much bigger fp32 network (~1–2 min). It never goes on the chip; it's only a training aid.

In [ ]:
# ✏️ TEACHER_SIZES: layer widths of the teacher.
TEACHER_SIZES = (16, 256, 256, 128, 5)

t0 = time.perf_counter()
teacher = train(mlp(TEACHER_SIZES), Xtr, ytr, epochs=4, seed=0)
record("teacher (fp32, big)", teacher, [(None, None)] * 4, sizes=TEACHER_SIZES, kind="teacher")
T_LOGITS = logits(teacher, Xtr)                  # the teacher's outputs on every training jet, computed once
print(f"({time.perf_counter()-t0:.0f} s)")

The distillation loss, in full. `out` are the student's outputs for a batch, `idx` the positions of those jets in the training set:
a mix of the usual cross-entropy with the true labels and a term that pulls the student's softmax towards the teacher's (KL divergence).

In [ ]:
# ✏️ ALPHA: weight on the true labels; (1 - ALPHA) goes to matching the teacher. 1.0 = no distillation.
ALPHA = 0.5

def kd_loss(out, idx):
    hard = F.cross_entropy(out, ytr[idx])                                                     # match the true labels
    soft = F.kl_div(F.log_softmax(out, 1), F.softmax(T_LOGITS[idx], 1), reduction="batchmean")   # match the teacher
    return ALPHA * hard + (1 - ALPHA) * soft

Now train small quantized students, each **twice**: once on the true labels only, once with the teacher. Same sizes, same bits, same data, same number of epochs.
Any difference comes from the teacher. (~1 min per training.)

In [ ]:
# ✏️ STUDENT_SIZES: student widths (first and last must stay 16 and 5).  STUDENT_BITS: their bits, every layer.
STUDENT_SIZES = [(16, 32, 16, 16, 5), (16, 16, 8, 8, 5)]
STUDENT_BITS = 6

scfg = [(STUDENT_BITS, STUDENT_BITS)] * 4
for sizes in STUDENT_SIZES:
    tag = "-".join(map(str, sizes[1:-1]))
    plain = train(qat_mlp(scfg, sizes), Xtr, ytr, epochs=4, seed=0)
    record(f"student {tag} {STUDENT_BITS}b", plain, scfg, sizes=sizes, kind="small")
    kd = train(qat_mlp(scfg, sizes), Xtr, ytr, epochs=4, seed=0, loss_fn=kd_loss)
    record(f"student {tag} {STUDENT_BITS}b + teacher", kd, scfg, sizes=sizes, kind="distilled")

In [ ]:
tags = ["-".join(map(str, s[1:-1])) for s in STUDENT_SIZES]
x = np.arange(len(tags))
plt.bar(x - 0.2, [RESULTS[f"student {t} {STUDENT_BITS}b"][1] / REJ32 for t in tags], 0.4, label="true labels only")
plt.bar(x + 0.2, [RESULTS[f"student {t} {STUDENT_BITS}b + teacher"][1] / REJ32 for t in tags], 0.4, label="with the teacher")
plt.axhline(1, c="gray", ls=":", label="fp32 baseline (64-32-32)")
plt.xticks(x, [f"hidden {t}" for t in tags]); plt.ylabel("top-tag rejection, fraction of fp32"); plt.title(f"{STUDENT_BITS}-bit students"); plt.legend(); plt.grid(alpha=.3, axis="y"); plt.show()

💭 **Think about:** Did the teacher help, and did it help the small student more or less than the bigger one? Try `ALPHA = 1.0` (no teacher) as a check, or `ALPHA = 0.2`.
Distillation costs nothing on the chip, only training time: if it helps, how many BOPs is that worth? You'll see in B5 where the students land relative to the other models.

## B5. Everything on one plot: the performance–BOPs front (20 min) ✋ Pause 🟩
Every model you scored today, from `RESULTS`. A model is **on the front** if no other model has both higher rejection *and* fewer BOPs.
The front is the menu of sensible designs: everything below it is beaten by something cheaper and better.

In [ ]:
def pareto(points):
    """points: list of (BOPs, rejection, name). Returns the ones on the front, cheapest first."""
    front, best = [], -np.inf
    for cost, rej, name in sorted(points, key=lambda p: (p[0], -p[1])):     # cheapest first; at equal cost, best first
        if rej > best:
            front.append((cost, rej, name)); best = rej
    return front

STYLE = {"fp32": "k*", "teacher": "kX", "PTQ": "o", "PTQ mixed": "D", "QAT": "s", "QAT mixed": "P", "pruned": "v", "small": "^", "distilled": "*"}
pts = [(v[2], v[1], k) for k, v in RESULTS.items() if v[3] != "teacher"]   # the teacher is a training aid, not a candidate
plt.figure(figsize=(10, 6))
for kind, mk in STYLE.items():
    sel = [(v[2], v[1]) for v in RESULTS.values() if v[3] == kind]
    if sel:
        plt.semilogx(*zip(*sel), mk, ms=8, ls="none", label=kind, alpha=.8)
fr = pareto(pts)
plt.semilogx([c for c, _, _ in fr], [r for _, r, _ in fr], "k-", lw=1, alpha=.5, drawstyle="steps-post", label="front")
for cost, rej, name in fr:
    plt.annotate(name, (cost, rej), fontsize=7, xytext=(4, 3), textcoords="offset points")
plt.xlabel("BOPs (log scale)"); plt.ylabel("top-tag background rejection @ 50% efficiency"); plt.title("Physics vs. hardware cost")
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=.3, which="both"); plt.show()
print("On the front, cheapest first:")
for cost, rej, name in fr:
    print(f"  {name:<34} BOPs {cost/1e3:9.1f}k | rejection {rej:6.0f} ({rej/REJ32:4.0%} of fp32)")

💭 **Think about:**
- Which *kinds* of model make it onto the front: PTQ, QAT, mixed, pruned, small, distilled? Which never do?
- Roughly how many times fewer BOPs than fp32 can you go while keeping, say, 80% of its rejection? 50%?
- Your FPGA has room for 50k BOPs. Which model on this plot would you deploy, and what would you try next to push the front up at that budget?
  (That is A2.)

✋ We'll put a few fronts from the room side by side.

## B6. Save a model for Lab 3 (5 min) 🟩
Lab 3 turns a quantized model into an FPGA design. Save one to your Google Drive (local Colab files vanish when the session ends).

In [ ]:
# ✏️ CHOSEN: the key of the model in `qat` to save (6, 8, 4, 3 or "mixed"). Unpruned QAT models only.
CHOSEN = 6

cfg = MIXED if CHOSEN == "mixed" else [(CHOSEN, CHOSEN)] * 4
ck = {"cfg": cfg, "sizes": SIZES, "bits": CHOSEN, "state_dict": qat[CHOSEN].state_dict()}
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # approve the pop-up once
    out_dir = "/content/drive/MyDrive/edgeml_models"
else:
    out_dir = "my_models"
os.makedirs(out_dir, exist_ok=True)
path = f"{out_dir}/lab2_model.pt"; torch.save(ck, path)
print("saved", path, "| config", fmt(cfg))